In [1]:
from pdf2image import convert_from_path
import os

In [2]:
os.makedirs("pages", exist_ok=True)

In [3]:
pdf_path = "input/model_answer.pdf"

In [4]:
if not os.path.exists(pdf_path):
    print("❌ PDF file not found. Check filename.")
else:
    pages = convert_from_path(pdf_path, dpi=300)

    for i, page in enumerate(pages):
        page_path = f"pages/model_page_{i+1}.jpg"
        page.save(page_path, "JPEG")
        print(f"Saved: {page_path}")

    print("✅ PDF successfully converted to images!")

Saved: pages/model_page_1.jpg
Saved: pages/model_page_2.jpg
Saved: pages/model_page_3.jpg
Saved: pages/model_page_4.jpg
✅ PDF successfully converted to images!


In [5]:
# STEP: Convert Student PDF to Images

from pdf2image import convert_from_path
import os

os.makedirs("pages_student", exist_ok=True)

student_pdf_path = "input/student_answer.pdf"

pages = convert_from_path(student_pdf_path, dpi=300)

for i, page in enumerate(pages):
    page_path = f"pages_student/student_page_{i+1}.jpg"
    page.save(page_path, "JPEG")
    print(f"Saved: {page_path}")

print("✅ Student PDF converted to images!")


Saved: pages_student/student_page_1.jpg
Saved: pages_student/student_page_2.jpg
Saved: pages_student/student_page_3.jpg
Saved: pages_student/student_page_4.jpg
✅ Student PDF converted to images!


In [8]:
# STEP 2: OCR Text Extraction

import pytesseract
from PIL import Image
import os

extracted_text = ""

# Read all images from pages folder
for file in sorted(os.listdir("pages")):
    if file.endswith(".jpg"):
        image_path = os.path.join("pages", file)
        img = Image.open(image_path)
        
        text = pytesseract.image_to_string(img)
        extracted_text += text + "\n"
        
        print(f"OCR done for: {file}")

print("✅ OCR completed for all pages!")


OCR done for: model_page_1.jpg
OCR done for: model_page_2.jpg
OCR done for: model_page_3.jpg
OCR done for: model_page_4.jpg
✅ OCR completed for all pages!


In [9]:
# STEP: OCR for Student Answer

import pytesseract
from PIL import Image

student_text = ""

for file in sorted(os.listdir("pages_student")):
    if file.endswith(".jpg"):
        image_path = os.path.join("pages_student", file)
        img = Image.open(image_path)
        
        text = pytesseract.image_to_string(img)
        student_text += text + "\n"
        
        print(f"OCR done for: {file}")

print("✅ OCR completed for student pages!")


OCR done for: student_page_1.jpg
OCR done for: student_page_2.jpg
OCR done for: student_page_3.jpg
OCR done for: student_page_4.jpg
✅ OCR completed for student pages!


In [10]:
# Save extracted text to file

os.makedirs("extracted_text", exist_ok=True)

with open("extracted_text/model_text.txt", "w") as f:
    f.write(extracted_text)

print("✅ Extracted text saved successfully!")


✅ Extracted text saved successfully!


In [11]:
# Save student extracted text

os.makedirs("extracted_text", exist_ok=True)

with open("extracted_text/student_text.txt", "w") as f:
    f.write(student_text)

print("✅ student_text.txt saved successfully!")


✅ student_text.txt saved successfully!


In [12]:
import re

# Load model text
with open("extracted_text/model_text.txt", "r") as f:
    model_text = f.read()

# Load student text
with open("extracted_text/student_text.txt", "r") as f:
    student_text = f.read()


# Function to split Q1, Q2 format
def split_answers(text):
    parts = re.split(r'(Q\d+)', text)
    answers = {}
    
    for i in range(1, len(parts), 2):
        question = parts[i]
        answer = parts[i+1].strip()
        answers[question] = answer
        
    return answers


model_answers = split_answers(model_text)
student_answers = split_answers(student_text)

print("Model Questions:", model_answers.keys())
print("Student Questions:", student_answers.keys())


Model Questions: dict_keys([])
Student Questions: dict_keys([])


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

results = {}

for question in model_answers:
    if question in student_answers:
        
        vectorizer = TfidfVectorizer()
        
        texts = [model_answers[question], student_answers[question]]
        tfidf_matrix = vectorizer.fit_transform(texts)
        
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        
        results[question] = similarity
        
        print(f"{question} Similarity: {round(similarity, 2)}")


In [15]:
max_marks = 5
marks = {}

for question, similarity in results.items():
    score = round(similarity * max_marks, 2)
    marks[question] = score
    print(f"{question} Marks: {score}/{max_marks}")
